# 04 — Validation Loop: Robustness Checks

Validates that findings from notebooks 01–03 are robust and not artifacts of parameter choices, corpus imbalance, or random chance.

In [1]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing .git")


PROJECT_ROOT = find_project_root(Path.cwd())
MODULE_ROOT = PROJECT_ROOT / "1a_BERTopic"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

os.environ["NUMBA_CACHE_DIR"] = str(PROJECT_ROOT / ".numba_cache")

OUTPUT_DIR = PROJECT_ROOT / "experiments" / "agenda_distortion" / "outputs"

import importlib
from bertopic_config import BERTopicConfig
from bertopic_pipeline import build_topic_model, build_embedding_model, prepare_documents
import merged_outlets_analysis as moa
moa = importlib.reload(moa)

from scipy.spatial.distance import jensenshannon
from sklearn.metrics.pairwise import cosine_similarity

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
RANDOM_STATE = 42

In [2]:
# Reload prepared documents
prepared_by_outlet = moa.load_all_prepared_documents(PROJECT_ROOT)
mainstream_prepared = prepared_by_outlet["tagesschau"]
alt_parts = [prepared_by_outlet[key] for key in moa.ALT_MEDIA_OUTLET_KEYS]
alt_prepared = pd.concat(alt_parts, ignore_index=True)

alt_docs = alt_prepared["document"].tolist()
ms_docs = mainstream_prepared["document"].tolist()

print(f"Alt media:  {len(alt_docs):,} docs")
print(f"Mainstream: {len(ms_docs):,} docs")

Alt media:  14,183 docs
Mainstream: 6,272 docs


## 1. Parameter Sensitivity Analysis

Re-run JSD across a grid of `min_topic_size` values. If JSD is consistently high, the finding is robust.

In [3]:
ALIGN_THRESHOLD = 0.65
MIN_CLUSTER_SIZES = [10, 20, 50]

embedding_model = build_embedding_model(BERTopicConfig())


def quick_jsd(alt_model, ms_model, alt_docs, ms_docs, align_threshold=0.65):
    """Compute JSD between two fitted BERTopic models."""
    alt_ti = alt_model.get_topic_info()
    ms_ti = ms_model.get_topic_info()
    alt_ids = alt_ti.loc[alt_ti["Topic"] != -1, "Topic"].tolist()
    ms_ids = ms_ti.loc[ms_ti["Topic"] != -1, "Topic"].tolist()

    if not alt_ids or not ms_ids:
        return float("nan"), 0, 0

    alt_emb = alt_model._extract_embeddings(
        [" ".join(w for w, _ in alt_model.get_topic(t)) for t in alt_ids], method="document"
    )
    ms_emb = ms_model._extract_embeddings(
        [" ".join(w for w, _ in ms_model.get_topic(t)) for t in ms_ids], method="document"
    )
    cross_sim = cosine_similarity(alt_emb, ms_emb)

    # Align
    aligned = []
    used_ms = set()
    uid = 0
    for i, alt_id in enumerate(alt_ids):
        best_j = int(cross_sim[i].argmax())
        if float(cross_sim[i, best_j]) >= align_threshold and best_j not in used_ms:
            aligned.append((uid, alt_id, ms_ids[best_j]))
            used_ms.add(best_j)
        else:
            aligned.append((uid, alt_id, None))
        uid += 1
    for j, ms_id in enumerate(ms_ids):
        if j not in used_ms:
            aligned.append((uid, None, ms_id))
            uid += 1

    n_unified = len(aligned)
    alt_topics_assigned = pd.Series(alt_model.transform(alt_docs)[0])
    ms_topics_assigned = pd.Series(ms_model.transform(ms_docs)[0])
    alt_c = alt_topics_assigned[alt_topics_assigned != -1].value_counts()
    ms_c = ms_topics_assigned[ms_topics_assigned != -1].value_counts()

    a_vec = np.zeros(n_unified)
    m_vec = np.zeros(n_unified)
    for u, a, m in aligned:
        if a is not None:
            a_vec[u] = alt_c.get(a, 0)
        if m is not None:
            m_vec[u] = ms_c.get(m, 0)

    a_norm = a_vec / a_vec.sum() if a_vec.sum() > 0 else a_vec
    m_norm = m_vec / m_vec.sum() if m_vec.sum() > 0 else m_vec
    jsd_val = float(jensenshannon(a_norm, m_norm, base=2) ** 2)
    avg_sim = float(cross_sim.mean())
    return jsd_val, avg_sim, n_unified


sensitivity_results = []

for mcs in MIN_CLUSTER_SIZES:
    print(f"\nTraining with min_cluster_size={mcs}...")
    cfg = BERTopicConfig(
        calculate_probabilities=False,
        hdbscan_min_cluster_size=mcs,
        hdbscan_min_samples=5,
        umap_n_neighbors=25,
        random_state=RANDOM_STATE,
    )
    alt_m = build_topic_model(cfg, embedding_model=embedding_model)
    alt_m.fit_transform(alt_docs)
    ms_m = build_topic_model(cfg, embedding_model=embedding_model)
    ms_m.fit_transform(ms_docs)

    jsd_val, avg_sim, n_topics = quick_jsd(alt_m, ms_m, alt_docs, ms_docs)
    sensitivity_results.append({
        "min_cluster_size": mcs,
        "jsd": jsd_val,
        "avg_cosine_sim": avg_sim,
        "n_unified_topics": n_topics,
    })
    print(f"  JSD={jsd_val:.4f}, avg_sim={avg_sim:.3f}, topics={n_topics}")

sensitivity_df = pd.DataFrame(sensitivity_results)
display(sensitivity_df)

jsd_range = sensitivity_df["jsd"].max() - sensitivity_df["jsd"].min()
if jsd_range < 0.1:
    print(f"\n→ JSD range across settings: {jsd_range:.4f} — ROBUST")
else:
    print(f"\n→ JSD range across settings: {jsd_range:.4f} — FRAGILE, investigate further")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Training with min_cluster_size=10...


2026-03-21 13:03:35,788 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/444 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

2026-03-21 13:19:42,362 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-03-21 13:19:42,389 - BERTopic - Dimensionality - Completed ✓
2026-03-21 13:19:42,397 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-21 13:19:42,491 - BERTopic - Cluster - Completed ✓


  JSD=0.6092, avg_sim=0.206, topics=79


,min_cluster_size,jsd,avg_cosine_sim,n_unified_topics
0,10,0.517559,0.200244,306
1,20,0.522751,0.200476,173
2,50,0.609151,0.206496,79



→ JSD range across settings: 0.0916 — ROBUST


## 2. Corpus Size Robustness Check

Downsample the larger corpus to match the smaller one. Re-run JSD. If results hold, the finding is not an imbalance artifact.

In [ ]:
min_size = min(len(alt_docs), len(ms_docs))
rng = np.random.RandomState(RANDOM_STATE)

if len(alt_docs) > min_size:
    idx = rng.choice(len(alt_docs), size=min_size, replace=False)
    alt_docs_ds = [alt_docs[i] for i in sorted(idx)]
    ms_docs_ds = ms_docs
    print(f"Downsampled alt media: {len(alt_docs):,} → {min_size:,}")
else:
    idx = rng.choice(len(ms_docs), size=min_size, replace=False)
    ms_docs_ds = [ms_docs[i] for i in sorted(idx)]
    alt_docs_ds = alt_docs
    print(f"Downsampled mainstream: {len(ms_docs):,} → {min_size:,}")

cfg_ds = BERTopicConfig(
    calculate_probabilities=False,
    hdbscan_min_cluster_size=20,
    hdbscan_min_samples=5,
    umap_n_neighbors=25,
    random_state=RANDOM_STATE,
)

alt_m_ds = build_topic_model(cfg_ds, embedding_model=embedding_model)
alt_m_ds.fit_transform(alt_docs_ds)
ms_m_ds = build_topic_model(cfg_ds, embedding_model=embedding_model)
ms_m_ds.fit_transform(ms_docs_ds)

jsd_ds, avg_sim_ds, n_topics_ds = quick_jsd(alt_m_ds, ms_m_ds, alt_docs_ds, ms_docs_ds)
print(f"\nDownsampled JSD: {jsd_ds:.4f} (vs original from sensitivity grid)")
print(f"Avg cosine sim:  {avg_sim_ds:.3f}")
print(f"Unified topics:  {n_topics_ds}")

2026-03-21 13:19:42,602 - BERTopic - Embedding - Transforming documents to embeddings.


Downsampled alt media: 14,183 → 6,272


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

2026-03-21 13:20:20,739 - BERTopic - Embedding - Completed ✓
2026-03-21 13:20:20,739 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-21 13:20:27,361 - BERTopic - Dimensionality - Completed ✓
2026-03-21 13:20:27,362 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-21 13:20:27,435 - BERTopic - Cluster - Completed ✓
2026-03-21 13:20:27,437 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-21 13:20:30,844 - BERTopic - Representation - Completed ✓
2026-03-21 13:20:32,477 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

2026-03-21 13:21:09,114 - BERTopic - Embedding - Completed ✓
2026-03-21 13:21:09,118 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-21 13:21:15,883 - BERTopic - Dimensionality - Completed ✓
2026-03-21 13:21:15,884 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-21 13:21:15,949 - BERTopic - Cluster - Completed ✓
2026-03-21 13:21:15,951 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-21 13:21:19,092 - BERTopic - Representation - Completed ✓


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

2026-03-21 13:22:01,046 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-03-21 13:22:01,063 - BERTopic - Dimensionality - Completed ✓
2026-03-21 13:22:01,063 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-21 13:22:01,119 - BERTopic - Cluster - Completed ✓


Batches:   0%|          | 0/196 [00:00<?, ?it/s]

## 3. Null Model Comparison

Shuffle media-type labels randomly. Recompute JSD. The observed JSD should be substantially higher than the null. If not, the finding may be a statistical artifact.

In [ ]:
N_PERMUTATIONS = 50  # Keep low since each requires full BERTopic fit

all_docs_combined = alt_docs + ms_docs
n_alt = len(alt_docs)

null_jsds = []

for i in range(N_PERMUTATIONS):
    perm_idx = rng.permutation(len(all_docs_combined))
    perm_alt = [all_docs_combined[j] for j in perm_idx[:n_alt]]
    perm_ms = [all_docs_combined[j] for j in perm_idx[n_alt:]]

    cfg_null = BERTopicConfig(
        calculate_probabilities=False,
        hdbscan_min_cluster_size=20,
        hdbscan_min_samples=5,
        umap_n_neighbors=25,
        random_state=RANDOM_STATE + i,
    )
    null_alt = build_topic_model(cfg_null, embedding_model=embedding_model)
    null_alt.fit_transform(perm_alt)
    null_ms = build_topic_model(cfg_null, embedding_model=embedding_model)
    null_ms.fit_transform(perm_ms)

    null_jsd, _, _ = quick_jsd(null_alt, null_ms, perm_alt, perm_ms)
    null_jsds.append(null_jsd)
    if (i + 1) % 10 == 0:
        print(f"  Permutation {i + 1}/{N_PERMUTATIONS}: null JSD = {null_jsd:.4f}")

null_jsds = np.array(null_jsds)
null_mean = float(null_jsds.mean())
null_95 = float(np.percentile(null_jsds, 95))

# Load the observed JSD from sensitivity grid (min_cluster_size=20)
observed_jsd = sensitivity_df.loc[sensitivity_df["min_cluster_size"] == 20, "jsd"].values[0]

print(f"\nObserved JSD:     {observed_jsd:.4f}")
print(f"Null mean JSD:    {null_mean:.4f}")
print(f"Null 95th pctile: {null_95:.4f}")
if observed_jsd > null_95:
    print("→ Observed JSD exceeds null 95th percentile: SIGNAL (not artifact)")
else:
    print("→ Observed JSD within null range: may be ARTIFACT")

## 4. LLM-Based Topic Validation

For the top 5 over-represented alt-media topics, use Claude to assess coherence and agenda-distortion plausibility.

In [ ]:
import anthropic

# Load the top over-represented alt media topics from notebook 02 outputs
# (If not saved, re-derive from doc_topics)
with open(OUTPUT_DIR / "alt_media_topic_words.json") as f:
    alt_word_lists = json.load(f)

alt_doc_topics = pd.read_csv(OUTPUT_DIR / "alt_media_doc_topics.csv")
alt_non_outlier = alt_doc_topics.loc[alt_doc_topics["topic"] != -1]
top_5_alt_topics = alt_non_outlier["topic"].value_counts().head(5).index.tolist()

client = anthropic.Anthropic()
llm_results = []

for topic_id in top_5_alt_topics:
    words = alt_word_lists.get(str(topic_id), [])
    word_str = ", ".join(words[:15])

    prompt = (
        f"You are a computational social scientist. Here are the top words for a topic "
        f"found predominantly in German alternative media: {word_str}.\n"
        f"1. Is this a coherent topic? (yes/no + reason)\n"
        f"2. Does it plausibly reflect agenda distortion or a narrowed political focus "
        f"compared to mainstream coverage? (yes/no + reason)\n"
        f"3. What would you label this topic?\n"
        f"Respond in JSON with keys: coherent, coherent_reason, agenda_distortion, "
        f"agenda_distortion_reason, label."
    )

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}],
    )
    response_text = response.content[0].text

    try:
        # Try to parse JSON from response
        parsed = json.loads(response_text)
    except json.JSONDecodeError:
        # Try extracting JSON from markdown code block
        import re
        match = re.search(r"```(?:json)?\s*\n?(\{.*?\})\s*\n?```", response_text, re.DOTALL)
        parsed = json.loads(match.group(1)) if match else {"raw_response": response_text}

    parsed["topic_id"] = topic_id
    parsed["words"] = words[:15]
    llm_results.append(parsed)
    print(f"Topic {topic_id}: {parsed.get('label', 'N/A')} — coherent={parsed.get('coherent', 'N/A')}")

# Save results
with open(OUTPUT_DIR / "llm_topic_validation.json", "w") as f:
    json.dump(llm_results, f, ensure_ascii=False, indent=2)

print(f"\nSaved LLM validation to: {OUTPUT_DIR / 'llm_topic_validation.json'}")

## Final Validation Summary

| Check | Result | Interpretation |
|---|---|---|
| JSD across parameter grid | [stable/unstable] | [robust/fragile] |
| JSD vs null model | [higher/not higher] | [signal/artifact] |
| Downsampled corpus JSD | [holds/drops] | [real/imbalance artifact] |
| LLM topic coherence | [N/5 coherent] | [clean/noisy topics] |

**Fill in after running all cells. Transfer final assessments to `findings.md`.**